In [1]:
from pathlib import Path
import sys
import re
import string
import numpy as np
import pandas as pd
from bnlp import BasicTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import load_and_clean_dataset

DATASET_PATH = PROJECT_ROOT / "data" / "bangla_news.csv"
df_clean = load_and_clean_dataset(DATASET_PATH)

tokenizer = BasicTokenizer()

print("Cleaned dataset shape:", df_clean.shape)
print("First article:")
print(df_clean.loc[df_clean.index[0], "content"][:200])

c:\Users\supan\OneDrive\DIU Materials\9th Semester\Fall 26 - Midterm\CSE432-Natural Language Processing\Project\Bengali Keyword Extractor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cleaned dataset shape: (11844, 6)
First article:
ছবি: সংগৃহীত সিন নদীতে অলিম্পিকের উদ্বোধনী অনুষ্ঠান হয়েছিল। এই নদীর পানি একেবারে দূষণমুক্ত, তা প্রমাণে প্যারিসের মেয়র নিজেই পানিতে নেমে পড়েছিলেন। কিন্তু এত কিছুর পরও সিন নদীকে দূষণমুক্ত প্রমাণ করা 


In [2]:
def preprocess_text(text):
    if not isinstance(text, str):
        return []

    tokens = tokenizer(text)

    cleaned_tokens = [
        token for token in tokens
        if not all(
            char in string.punctuation or char in "।–—…"
            for char in token
        )
    ]

    cleaned_tokens = [
        token for token in cleaned_tokens
        if not re.fullmatch(r"\d+", token)
    ]

    return cleaned_tokens

In [3]:
sample_article = df_clean.loc[df_clean.index[0], "content"]
sample_tokens = preprocess_text(sample_article)

print(sample_tokens[:40])
print("Total tokens:", len(sample_tokens))

['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'হয়েছিল', 'এই', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'তা', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'নিজেই', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'কিন্তু', 'এত', 'কিছুর', 'পরও', 'সিন', 'নদীকে', 'দূষণমুক্ত', 'প্রমাণ', 'করা', 'গেল', 'না', 'দূষণের', 'কারণে', 'সিন', 'নদীতে', 'অনুষ্ঠিতব্য', 'অলিম্পিকের', 'ট্রায়াথলন', 'ইভেন্ট']
Total tokens: 90


In [4]:
# Quick experiment with 5 articles
sample_df = df_clean.head(5).copy()

sample_df["tokens"] = sample_df["content"].apply(preprocess_text)

sample_df["processed_text"] = sample_df["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print(sample_df[["category", "processed_text"]].head(2).to_string(index=False))

category                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          processed_text
  sports ছবি সংগৃহীত সিন নদীতে অলিম্পিকের উদ্বোধনী অনুষ্ঠান হয়েছিল এই নদীর পানি একেবারে দূষণমুক্ত তা প্রমাণে প্যারিসের মেয়র নিজেই পানিতে নেমে পড়েছিলেন কিন্তু এত কিছুর পরও সিন নদীকে দূষণমুক্ত প্রমাণ করা গেল না দূষণের কারণে সিন নদীতে অনুষ্ঠিতব্য অলিম্পিকের ট্রায়াথলন ইভেন্ট স্থগিত করা হয়েছে খবর দ্য গার্ডিয়ানের জানা গেছে সিন নদীর দূষণের ইতিহাস বেশ পুরোনো প্রায় বছর আগে সালে নদীটিতে সাঁতার কাটা নিষিদ্ধ করে দেয়া হয় নদী

In [5]:
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

tfidf_matrix = vectorizer.fit_transform(sample_df["processed_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (5, 276)
Vocabulary size: 276


In [6]:
feature_names = vectorizer.get_feature_names_out()

def extract_top_keywords(tfidf_matrix, feature_names, top_k=10):
    results = []

    for row_index in range(tfidf_matrix.shape[0]):
        scores = tfidf_matrix[row_index].toarray().flatten()

        top_indices = np.argsort(scores)[::-1][:top_k]

        keywords = [
            (feature_names[index], round(scores[index], 4))
            for index in top_indices
            if scores[index] > 0
        ]

        results.append(keywords)

    return results


sample_keywords = extract_top_keywords(
    tfidf_matrix,
    feature_names,
    top_k=10
)

In [7]:
for i, keywords in enumerate(sample_keywords):
    print(f"\nArticle {i + 1} ({sample_df.iloc[i]['category']}):")

    for word, score in keywords:
        print(f"  {word}: {score}")


Article 1 (sports):
  সিন: 0.3475
  নদীতে: 0.2607
  নদীর: 0.2607
  এই: 0.2103
  সাঁতার: 0.1738
  দূষণমুক্ত: 0.1738
  নদীকে: 0.1738
  পানিতে: 0.1738
  ট্রায়াথলন: 0.1738
  হয়েছিল: 0.1738

Article 2 (sports):
  ক্রিকেট: 0.4145
  এসিসির: 0.3109
  এশিয়ান: 0.2073
  মধ্যে: 0.2073
  সভাপতি: 0.1672
  হচ্ছে: 0.1672
  কন্ট্রোল: 0.1036
  এর: 0.1036
  সবচেয়ে: 0.1036
  এসিসি: 0.1036

Article 3 (sports):
  বিশ্বকাপ: 0.3589
  হবার: 0.2871
  ও: 0.2871
  পর্যন্ত: 0.2153
  আসর: 0.2153
  করেছে: 0.2153
  ফিফা: 0.2153
  হোস্ট: 0.1436
  স্পেন: 0.1436
  যৌথভাবে: 0.1436

Article 4 (sports):
  কোয়ার্টার: 0.2313
  চায়: 0.2313
  ইউক্রেনের: 0.2313
  রেসে: 0.2313
  হবে: 0.1549
  সম্ভাবনা: 0.1157
  সময়: 0.1157
  সেক্ষেত্রে: 0.1157
  সেই: 0.1157
  একটি: 0.1157

Article 5 (sports):
  ভারত: 0.3642
  টি: 0.3642
  আজ: 0.2939
  শ্রীলঙ্কার: 0.1821
  শ্রীলঙ্কা: 0.1821
  পদকের: 0.1821
  লাইভ: 0.1821
  ইভেন্টসকাল: 0.1821
  সন্ধ্যায়: 0.1821
  লড়াই: 0.1821


In [8]:
df_tfidf = df_clean.copy()

df_tfidf["tokens"] = df_tfidf["content"].apply(preprocess_text)

df_tfidf["processed_text"] = df_tfidf["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print("Articles processed:", len(df_tfidf))
print(
    "Articles with no tokens:",
    (df_tfidf["processed_text"].str.strip() == "").sum()
)

Articles processed: 11844
Articles with no tokens: 0


In [9]:
vectorizer_full = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=100000
)

tfidf_matrix_full = vectorizer_full.fit_transform(
    df_tfidf["processed_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix_full.shape)
print("Vocabulary size:", len(vectorizer_full.vocabulary_))

TF-IDF matrix shape: (11844, 59661)
Vocabulary size: 59661


In [10]:
feature_names_full = vectorizer_full.get_feature_names_out()

top_keywords_full = extract_top_keywords(
    tfidf_matrix_full,
    feature_names_full,
    top_k=10
)

# Display the top keywords for the first 5 articles
for i in range(5):
    print(
        f"\nArticle {i + 1} "
        f"({df_tfidf.iloc[i]['category']}):"
    )
    print("Title:", df_tfidf.iloc[i]["title"])

    for word, score in top_keywords_full[i]:
        print(f"  {word}: {score}")


Article 1 (sports):
Title: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট
  সিন: 0.4187
  নদীতে: 0.2944
  নদীর: 0.2608
  ট্রায়াথলন: 0.2478
  নদীকে: 0.2375
  দূষণমুক্ত: 0.2375
  দূষণের: 0.2244
  সাঁতার: 0.202
  অলিম্পিকের: 0.1843
  কাটা: 0.1794

Article 2 (sports):
Title: এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!
  এসিসির: 0.4696
  ক্রিকেট: 0.26
  এশিয়ান: 0.2508
  সভাপতি: 0.1696
  কাউন্সিলেরও: 0.1634
  লড়বেন: 0.1634
  চেয়ারম্যানই: 0.1634
  নাকভির: 0.1634
  ইন্ডিয়ার: 0.1422
  এসিসি: 0.1422

Article 3 (sports):
Title: ২০৩০ ও ২০৩৪ বিশ্বকাপের হোস্ট জানা যাবে দুইদিন পর!
  হবার: 0.3004
  বিশ্বকাপ: 0.284
  বিড: 0.2519
  ডেটলাইন: 0.2519
  ফিফা: 0.2433
  হোস্ট: 0.2414
  মরক্কো: 0.2281
  আসর: 0.193
  পর্তুগাল: 0.1896
  যৌথভাবে: 0.1682

Article 4 (sports):
Title: প্যারিস অলিম্পিক: কোয়ার্টার ফাইনালে ওঠার লক্ষ্যে রাতে মাঠে নামবে আর্জেন্টিনা
  রেসে: 0.326
  কোয়ার্টার: 0.2544
  ইউক্রেনের: 0.2349
  চায়: 0.2327
  মাশ্চেরানোর: 0.1857
  লিও: 0.1571
  ম্যানেজমেন্ট: 0.1514
  সেক্ষেত্রে: 0.1514
 

In [11]:
bangla_stopwords = {
    "এই", "ওই", "তা", "এটা", "সেটা",
    "এবং", "আর", "কিন্তু", "তবে",
    "এর", "এতে", "একে", "তাকে",
    "যে", "যা", "যার", "যেখানে",
    "কী", "কে", "কেন", "করে", "করা",
    "হয়", "হয়েছে", "হয়েছিল", "হবে",
    "হচ্ছে", "ছিল", "ছিলেন",
    "না", "নয়", "নেই",
    "জন্য", "থেকে", "সঙ্গে", "মধ্যে",
    "পর", "পরে", "আগে", "আরও",
    "অনেক", "বেশি", "কম", "সব",
    "নিজেই", "নিজের", "তাদের", "তার",
    "একটি", "এক", "দুই", "কোনো",
    "প্রায়", "তাই", "যদি", "তাহলে",
}

In [12]:
def remove_stopwords(tokens, stopwords):
    return [
        token for token in tokens
        if token not in stopwords
    ]


sample_tokens = df_tfidf.iloc[0]["tokens"]

print("Before:", sample_tokens[:30])

filtered_tokens = remove_stopwords(
    sample_tokens,
    bangla_stopwords
)

print("\nAfter:", filtered_tokens[:30])
print("Tokens before:", len(sample_tokens))
print("Tokens after:", len(filtered_tokens))

Before: ['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'হয়েছিল', 'এই', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'তা', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'নিজেই', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'কিন্তু', 'এত', 'কিছুর', 'পরও', 'সিন', 'নদীকে', 'দূষণমুক্ত', 'প্রমাণ', 'করা']

After: ['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'এত', 'কিছুর', 'পরও', 'সিন', 'নদীকে', 'দূষণমুক্ত', 'প্রমাণ', 'গেল', 'দূষণের', 'কারণে', 'সিন', 'নদীতে', 'অনুষ্ঠিতব্য']
Tokens before: 90
Tokens after: 70


In [13]:
from bnlp import BengaliCorpus as corpus

bnlp_stopwords = set(corpus.stopwords)

print("Our stop-word count:", len(bangla_stopwords))
print("BNLP stop-word count:", len(bnlp_stopwords))

print("\nWords in both lists:")
print(sorted(bangla_stopwords & bnlp_stopwords))

print("\nWords only in our list:")
print(sorted(bangla_stopwords - bnlp_stopwords)[:30])

print("\nWords only in BNLP's list:")
print(sorted(bnlp_stopwords - bangla_stopwords)[:30])

Our stop-word count: 56
BNLP stop-word count: 398

Words in both lists:
['অনেক', 'আগে', 'আর', 'আরও', 'এই', 'একটি', 'একে', 'এটা', 'এতে', 'এবং', 'এর', 'ওই', 'করা', 'করে', 'কিন্তু', 'কী', 'কে', 'কেন', 'কোনো', 'ছিল', 'ছিলেন', 'জন্য', 'তবে', 'তা', 'তাই', 'তাকে', 'তাদের', 'তার', 'তাহলে', 'থেকে', 'দুই', 'নয়', 'না', 'নিজেই', 'নিজের', 'নেই', 'পর', 'পরে', 'প্রায়', 'বেশি', 'মধ্যে', 'যদি', 'যা', 'যার', 'যে', 'যেখানে', 'সঙ্গে', 'সব', 'সেটা', 'হচ্ছে', 'হবে', 'হয়', 'হয়েছিল', 'হয়েছে']

Words only in our list:
['এক', 'কম']

Words only in BNLP's list:
['অতএব', 'অথচ', 'অথবা', 'অনুযায়ী', 'অনেকে', 'অনেকেই', 'অন্তত', 'অন্য', 'অবধি', 'অবশ্য', 'অর্থাত', 'আই', 'আগামী', 'আগেই', 'আছে', 'আজ', 'আদ্যভাগে', 'আপনার', 'আপনি', 'আবার', 'আমরা', 'আমাকে', 'আমাদের', 'আমার', 'আমি', 'ই', 'ইত্যাদি', 'ইহা', 'উচিত', 'উত্তর']


In [14]:
bnlp_filtered_tokens = remove_stopwords(
    sample_tokens,
    bnlp_stopwords
)

print("Original token count:", len(sample_tokens))
print("Our list:", len(filtered_tokens))
print("BNLP list:", len(bnlp_filtered_tokens))

print("\nBNLP-filtered tokens:")
print(bnlp_filtered_tokens[:40])

Original token count: 90
Our list: 70
BNLP list: 62

BNLP-filtered tokens:
['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'কিছুর', 'পরও', 'সিন', 'নদীকে', 'দূষণমুক্ত', 'প্রমাণ', 'দূষণের', 'কারণে', 'সিন', 'নদীতে', 'অনুষ্ঠিতব্য', 'অলিম্পিকের', 'ট্রায়াথলন', 'ইভেন্ট', 'স্থগিত', 'খবর', 'দ্য', 'গার্ডিয়ানের', 'সিন', 'নদীর', 'দূষণের', 'ইতিহাস', 'পুরোনো']


In [15]:
df_tfidf["tokens_no_stopwords"] = df_tfidf["tokens"].apply(
    lambda tokens: remove_stopwords(tokens, bnlp_stopwords)
)

df_tfidf["processed_no_stopwords"] = (
    df_tfidf["tokens_no_stopwords"].apply(
        lambda tokens: " ".join(tokens)
    )
)

print("Articles:", len(df_tfidf))
print(
    "Articles with no tokens after filtering:",
    (df_tfidf["processed_no_stopwords"].str.strip() == "").sum()
)

print("\nBefore:")
print(df_tfidf.iloc[0]["tokens"][:25])

print("\nAfter:")
print(df_tfidf.iloc[0]["tokens_no_stopwords"][:25])

Articles: 11844
Articles with no tokens after filtering: 0

Before:
['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'হয়েছিল', 'এই', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'তা', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'নিজেই', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'কিন্তু', 'এত', 'কিছুর', 'পরও']

After:
['ছবি', 'সংগৃহীত', 'সিন', 'নদীতে', 'অলিম্পিকের', 'উদ্বোধনী', 'অনুষ্ঠান', 'নদীর', 'পানি', 'একেবারে', 'দূষণমুক্ত', 'প্রমাণে', 'প্যারিসের', 'মেয়র', 'পানিতে', 'নেমে', 'পড়েছিলেন', 'কিছুর', 'পরও', 'সিন', 'নদীকে', 'দূষণমুক্ত', 'প্রমাণ', 'দূষণের', 'কারণে']


In [16]:
vectorizer_stopwords = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=100000
)

tfidf_matrix_stopwords = vectorizer_stopwords.fit_transform(
    df_tfidf["processed_no_stopwords"]
)

print("TF-IDF matrix shape:", tfidf_matrix_stopwords.shape)
print("Vocabulary size:", len(vectorizer_stopwords.vocabulary_))

TF-IDF matrix shape: (11844, 59295)
Vocabulary size: 59295


In [17]:
stopword_features = vectorizer_stopwords.get_feature_names_out()

top_keywords_stopwords = extract_top_keywords(
    tfidf_matrix_stopwords,
    stopword_features,
    top_k=10
)

print("TITLE:", df_tfidf.iloc[0]["title"])

print("\nBaseline keywords:")
for word, score in top_keywords_full[0]:
    print(f"{word}: {score}")

print("\nWith BNLP stop words:")
for word, score in top_keywords_stopwords[0]:
    print(f"{word}: {score}")

TITLE: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট

Baseline keywords:
সিন: 0.4187
নদীতে: 0.2944
নদীর: 0.2608
ট্রায়াথলন: 0.2478
নদীকে: 0.2375
দূষণমুক্ত: 0.2375
দূষণের: 0.2244
সাঁতার: 0.202
অলিম্পিকের: 0.1843
কাটা: 0.1794

With BNLP stop words:
সিন: 0.4368
নদীতে: 0.3072
নদীর: 0.2721
ট্রায়াথলন: 0.2586
দূষণমুক্ত: 0.2477
নদীকে: 0.2477
দূষণের: 0.2341
সাঁতার: 0.2107
অলিম্পিকের: 0.1922
কাটা: 0.1872


In [18]:
article_index = 1

print("TITLE:", df_tfidf.iloc[article_index]["title"])

print("\nBaseline keywords:")
for word, score in top_keywords_full[article_index]:
    print(f"{word}: {score}")

print("\nWith BNLP stop words:")
for word, score in top_keywords_stopwords[article_index]:
    print(f"{word}: {score}")

TITLE: এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!

Baseline keywords:
এসিসির: 0.4696
ক্রিকেট: 0.26
এশিয়ান: 0.2508
সভাপতি: 0.1696
কাউন্সিলেরও: 0.1634
লড়বেন: 0.1634
চেয়ারম্যানই: 0.1634
নাকভির: 0.1634
ইন্ডিয়ার: 0.1422
এসিসি: 0.1422

With BNLP stop words:
এসিসির: 0.4889
ক্রিকেট: 0.2707
এশিয়ান: 0.2611
সভাপতি: 0.1766
লড়বেন: 0.1701
নাকভির: 0.1701
কাউন্সিলেরও: 0.1701
চেয়ারম্যানই: 0.1701
ইন্ডিয়ার: 0.1481
এসিসি: 0.1481


In [19]:
from src.keyword_extraction import load_dataset, clean_text, tokenize

df_test = load_dataset()

print("Dataset shape:", df_test.shape)
print("First title:", df_test.iloc[0]["title"])

sample_text = (
    "নাদাল-জোকোভিচ ম্যাচে Djokovic 6-1, 6-4 সেটে জিতেছেন। "
    "US Open 2024!"
)

print("\nCleaned text:", clean_text(sample_text, keep_latin=True))
print("\nTokens:", tokenize(sample_text))

Dataset shape: (11844, 6)
First title: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট

Cleaned text: নাদাল-জোকোভিচ ম্যাচে Djokovic 6-1, 6-4 সেটে জিতেছেন। US Open 2024!

Tokens: ['নাদাল', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', 'সেটে', 'জিতেছেন', 'US', 'Open']


In [20]:
from src.keyword_extraction import preprocess

sample_text = (
    "নাদাল-জোকোভিচ ম্যাচে Djokovic 6-1, 6-4 সেটে জিতেছেন। "
    "US Open 2024!"
)

print(preprocess(sample_text))

['নাদাল', 'জোকোভিচ', 'ম্যাচে', 'Djokovic', 'সেটে', 'জিতেছেন', 'US', 'Open']


In [21]:
from src.keyword_extraction import extract_keywords_tfidf

test_documents = [
    "সিন নদী দূষণ অলিম্পিক ট্রায়াথলন",
    "ক্রিকেট অলিম্পিক ক্রিকেট দল",
    "সিন নদী দূষণ পরিবেশ",
]

test_results = extract_keywords_tfidf(
    test_documents,
    top_n=3,
)

for i, keywords in enumerate(test_results, start=1):
    print(f"\nDocument {i}:")
    for word, score in keywords:
        print(f"  {word}: {score:.4f}")


Document 1:
  ট্রায়াথলন: 0.5494
  সিন: 0.4178
  দূষণ: 0.4178

Document 2:
  ক্রিকেট: 0.8468
  দল: 0.4234
  অলিম্পিক: 0.3220

Document 3:
  পরিবেশ: 0.6047
  সিন: 0.4599
  নদী: 0.4599


In [22]:
from src.keyword_extraction import (
    load_dataset,
    preprocess,
    extract_keywords_tfidf,
    save_results,
)

In [23]:
df_final = load_dataset()

df_final["tokens"] = df_final["content"].apply(preprocess)

df_final["processed_text"] = df_final["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print("Articles:", len(df_final))
print(
    "Articles with no tokens:",
    (df_final["processed_text"].str.strip() == "").sum()
)

Articles: 11844
Articles with no tokens: 0


In [24]:
documents = df_final["processed_text"].tolist()

keywords = extract_keywords_tfidf(
    documents,
    top_n=10,
    max_features=100000,
)

print("Articles with keyword results:", len(keywords))
print("\nFirst article:")
print("Title:", df_final.iloc[0]["title"])

for word, score in keywords[0]:
    print(f"{word}: {score:.4f}")

Articles with keyword results: 11844

First article:
Title: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট
সিন: 0.4402
নদীতে: 0.3095
নদীর: 0.2742
ট্রায়াথলন: 0.2496
দূষণমুক্ত: 0.2496
নদীকে: 0.2496
দূষণের: 0.2359
সাঁতার: 0.2123
অলিম্পিকের: 0.1937
কাটা: 0.1886


In [25]:
results_df = pd.DataFrame({
    "title": df_final["title"].values,
    "category": df_final["category"].values,
    "url": df_final["url"].values,
    "keywords": [
        ", ".join(word for word, score in article_keywords)
        for article_keywords in keywords
    ],
    "keyword_scores": [
        "; ".join(
            f"{word}:{score:.4f}"
            for word, score in article_keywords
        )
        for article_keywords in keywords
    ],
})

output_path = save_results(results_df)

print("Results saved to:", output_path.resolve())
print("Output shape:", results_df.shape)
print(results_df.head(3).to_string(index=False))

Results saved to: C:\Users\supan\OneDrive\DIU Materials\9th Semester\Fall 26 - Midterm\CSE432-Natural Language Processing\Project\Bengali Keyword Extractor\output\keywords.csv
Output shape: (11844, 5)
                                                  title category                           url                                                                                   keywords                                                                                                                                                   keyword_scores
সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট   sports https://jamuna.tv/news/552127           সিন, নদীতে, নদীর, ট্রায়াথলন, দূষণমুক্ত, নদীকে, দূষণের, সাঁতার, অলিম্পিকের, কাটা           সিন:0.4402; নদীতে:0.3095; নদীর:0.2742; ট্রায়াথলন:0.2496; দূষণমুক্ত:0.2496; নদীকে:0.2496; দূষণের:0.2359; সাঁতার:0.2123; অলিম্পিকের:0.1937; কাটা:0.1886
                 এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!   sports https://jamuna.tv/news/552123 এসিসির, ক্

In [26]:
print(df_final["category"].value_counts())

category
sports           2976
international    2972
entertainment    2961
national         2935
Name: count, dtype: int64


In [27]:
from pathlib import Path
import pandas as pd

# Number of articles to manually evaluate from each category
allocation = {
    "sports": 13,
    "international": 13,
    "entertainment": 12,
    "national": 12,
}

# Preserve original row indices so predictions can be matched later
df_eval_base = df_final.copy()
df_eval_base["article_id"] = df_eval_base.index

# Draw a reproducible sample from each category
sampled_parts = []

for category, sample_size in allocation.items():
    category_df = df_eval_base[
        df_eval_base["category"] == category
    ]

    sampled_parts.append(
        category_df.sample(
            n=sample_size,
            random_state=42
        )
    )

df_eval = pd.concat(sampled_parts).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Create a separate file for human annotations.
# Deliberately exclude predicted keywords to avoid bias.
annotation_df = df_eval[
    ["article_id", "category", "title", "content"]
].copy()

annotation_df["reference_keywords"] = ""

output_path = Path("../output/manual_evaluation.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

annotation_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Sample size:", len(annotation_df))
print("\nSample distribution:")
print(annotation_df["category"].value_counts())
print("\nSaved to:", output_path.resolve())

Sample size: 50

Sample distribution:
category
international    13
sports           13
national         12
entertainment    12
Name: count, dtype: int64

Saved to: C:\Users\supan\OneDrive\DIU Materials\9th Semester\Fall 26 - Midterm\CSE432-Natural Language Processing\Project\Bengali Keyword Extractor\output\manual_evaluation.csv


In [34]:
# Load manually annotated reference keywords
annotation_path = PROJECT_ROOT / "output" / "manual_evaluation.csv"

manual_df = pd.read_csv(
    annotation_path,
    encoding="utf-8-sig"
)

# Reconstruct predictions, preserving the original article IDs
prediction_df = pd.DataFrame({
    "article_id": df_final.index,
    "predicted_keywords": [
        [word for word, score in article_keywords]
        for article_keywords in keywords
    ]
})

# Match predictions to the corresponding annotated articles
evaluation_df = manual_df.merge(
    prediction_df,
    on="article_id",
    how="left",
    validate="one_to_one"
)

print("Annotated articles:", len(manual_df))
print("Matched predictions:", evaluation_df["predicted_keywords"].notna().sum())
print("Missing reference annotations:",
      evaluation_df["reference_keywords"].isna().sum())
print("Empty reference annotations:",
      evaluation_df["reference_keywords"].fillna("").str.strip().eq("").sum())

display(
    evaluation_df[
        ["article_id", "category", "title", "reference_keywords"]
    ].head(3)
)

Annotated articles: 50
Matched predictions: 50
Missing reference annotations: 0
Empty reference annotations: 0


,article_id,category,title,reference_keywords
0,5380,international,মিউং : হতদরিদ্র পরিবার থেকে দেশের শীর্ষ নেতা,মিউং; ছুরিকাঘাত; চিকিৎসক; রয়টার্স; কোরিয়ার; ডে...
1,11327,national,শুরু হলো পবিত্র মাহে রমজান,রমজান; কুরআন; তারাবির; নামাজ; সেহরি; সিয়াম; মু...
2,8654,entertainment,শিল্পী সমিতির ভোট: ফল নিয়ে চলছে নানা গুঞ্জন,শিল্পী; সমিতির; ভোট; ফল; গুঞ্জন; নির্বাচন; এফড...


In [35]:
METRIC_COLUMNS = ["precision_at_10", "recall_at_10", "f1_at_10", "matched_keywords"]

def normalize_keyword(keyword):
    return re.sub(r"\s+", " ", str(keyword)).strip().casefold()

def parse_reference_keywords(text):
    if pd.isna(text):
        return set()
    return {
        normalize_keyword(word)
        for word in str(text).split(";")
        if normalize_keyword(word)
    }

def calculate_metrics(reference_text, predicted_keywords):
    reference = parse_reference_keywords(reference_text)
    predicted = {normalize_keyword(word) for word in list(predicted_keywords)[:10]}

    # Avoid silently scoring articles with missing annotations
    if not reference:
        return pd.Series({
            "precision_at_10": np.nan,
            "recall_at_10": np.nan,
            "f1_at_10": np.nan,
            "matched_keywords": 0,
        })

    relevant = len(reference & predicted)
    precision = relevant / 10
    recall = relevant / len(reference)
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return pd.Series({
        "precision_at_10": precision,
        "recall_at_10": recall,
        "f1_at_10": f1,
        "matched_keywords": relevant,
    })

# FIX: make this cell safe to re-run — drop metric columns from a previous run
evaluation_df = evaluation_df.drop(columns=METRIC_COLUMNS, errors="ignore")

metric_columns = evaluation_df.apply(
    lambda row: calculate_metrics(
        row["reference_keywords"],
        row["predicted_keywords"]
    ),
    axis=1
)

evaluation_df = pd.concat([evaluation_df, metric_columns], axis=1)

overall_metrics = evaluation_df[METRIC_COLUMNS[:3]].mean() * 100

print("OVERALL RESULTS (Macro-average)")
print(f"Precision@10: {overall_metrics['precision_at_10']:.2f}%")
print(f"Recall@10:    {overall_metrics['recall_at_10']:.2f}%")
print(f"F1@10:        {overall_metrics['f1_at_10']:.2f}%")

print("\nAverage matched keywords per article:",
      f"{evaluation_df['matched_keywords'].mean():.2f} / 10")

OVERALL RESULTS (Macro-average)
Precision@10: 32.60%
Recall@10:    32.60%
F1@10:        32.60%

Average matched keywords per article: 3.26 / 10


In [36]:
category_results = (
    evaluation_df
    .groupby("category")[
        ["precision_at_10", "recall_at_10", "f1_at_10"]
    ]
    .mean()
    .mul(100)
    .round(2)
)

display(category_results)

,precision_at_10,recall_at_10,f1_at_10
category,,,
entertainment,29.17,29.17,29.17
international,32.31,32.31,32.31
national,34.17,34.17,34.17
sports,34.62,34.62,34.62


In [39]:
for _, row in evaluation_df.head(5).iterrows():
    print("=" * 80)
    print("CATEGORY:", row["category"])
    print("TITLE:", row["title"])
    print("\nREFERENCE KEYWORDS:")
    print(row["reference_keywords"])
    print("\nPREDICTED KEYWORDS:")
    print(", ".join(row["predicted_keywords"][:10]))
    print("MATCHED:", row["matched_keywords"], "/ 10")
    print()

CATEGORY: international
TITLE: মিউং : হতদরিদ্র পরিবার থেকে দেশের শীর্ষ নেতা

REFERENCE KEYWORDS:
মিউং; ছুরিকাঘাত; চিকিৎসক; রয়টার্স; কোরিয়ার; ডেমোক্রেটিক; পার্টির; রাজনীতিক; আইনজীবী; হতদরিদ্র

PREDICTED KEYWORDS:
রাত্রিকালীন, মিউংয়ের, হতদরিদ্র, পড়াশুনার, মিউং, অর্থাভাবে, মিউংকে, যোগাতে, অনুপ্রেরণার, জায়ে
MATCHED: 2.0 / 10

CATEGORY: national
TITLE: শুরু হলো পবিত্র মাহে রমজান

REFERENCE KEYWORDS:
রমজান; কুরআন; তারাবির; নামাজ; সেহরি; সিয়াম; মুসল্লিরা; মসজিদ; চাঁদ; আল্লাহর

PREDICTED KEYWORDS:
মুসল্লিরা, তারাবি, তারাবির, পবিত্র, ধর্মপ্রাণ, মাস, নামাজ, রমজান, পড়তে, মোকাররমেও
MATCHED: 4.0 / 10

CATEGORY: entertainment
TITLE: শিল্পী সমিতির ভোট: ফল নিয়ে চলছে নানা গুঞ্জন

REFERENCE KEYWORDS:
শিল্পী; সমিতির; ভোট; ফল; গুঞ্জন; নির্বাচন; এফডিসি; প্রার্থী; সভাপতি; সাধারণ

PREDICTED KEYWORDS:
মিশা, জায়েদ, অপেক্ষামানদের, কাঞ্চনের, গুঞ্জনও, পাল্লা, ফলাফলের, হাড্ডাহাড্ডি, ঘোষণায়, এফডিসি
MATCHED: 1.0 / 10

CATEGORY: national
TITLE: চাকরিতে পিতৃত্বকালীন ছুটি কেন নয়, হাইকোর্টের রুল

REFERENCE KEYWORD

In [40]:
print("Dataset shape:", df_final.shape)
print("Columns:", df_final.columns.tolist())

display(
    df_final[["title", "content"]].head(3)
)

print("\nCurrent predictions:")
for i in range(3):
    print("\nTitle:", df_final.iloc[i]["title"])
    print(", ".join(word for word, score in keywords[i]))

Dataset shape: (11844, 8)
Columns: ['title', 'published_date', 'reporter', 'category', 'url', 'content', 'tokens', 'processed_text']


,title,content
0,"সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথ...",ছবি: সংগৃহীত সিন নদীতে অলিম্পিকের উদ্বোধনী অনু...
1,এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!,২০২৫ সালের জানুয়ারিতে এশিয়ান ক্রিকেট কাউন্সি...
2,২০৩০ ও ২০৩৪ বিশ্বকাপের হোস্ট জানা যাবে দুইদিন পর!,ফিফা বিশ্বকাপ ২০৩০ ও ৩৪ সালের স্বাগতিক হবার লড়...



Current predictions:

Title: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট
সিন, নদীতে, নদীর, ট্রায়াথলন, দূষণমুক্ত, নদীকে, দূষণের, সাঁতার, অলিম্পিকের, কাটা

Title: এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!
এসিসির, ক্রিকেট, এশিয়ান, সভাপতি, চেয়ারম্যানই, নাকভির, কাউন্সিলেরও, এসিসি, মহসিন, পিসিবির

Title: ২০৩০ ও ২০৩৪ বিশ্বকাপের হোস্ট জানা যাবে দুইদিন পর!
হবার, বিশ্বকাপ, বিড, ডেটলাইন, ফিফা, হোস্ট, মরক্কো, আসর, পর্তুগাল, যৌথভাবে


In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_title_weighted_documents(df):
    documents = []

    for _, row in df.iterrows():
        title_tokens = preprocess(row["title"])
        content_tokens = preprocess(row["content"])

        # Give title terms extra weight
        weighted_tokens = title_tokens * 3 + content_tokens

        documents.append(" ".join(weighted_tokens))

    return documents


# Create separate experimental documents
weighted_documents = build_title_weighted_documents(df_final)

# Fit a separate vectorizer; do not overwrite the baseline
title_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=100000,
)

title_tfidf_matrix = title_vectorizer.fit_transform(weighted_documents)

title_weighted_keywords = extract_top_keywords(
    title_tfidf_matrix,
    title_vectorizer.get_feature_names_out(),
    top_k=10,
)

print("Articles:", len(title_weighted_keywords))
print("Matrix shape:", title_tfidf_matrix.shape)

for i in range(3):
    print("\nTitle:", df_final.iloc[i]["title"])
    print(", ".join(
        word for word, score in title_weighted_keywords[i]
    ))

Articles: 11844
Matrix shape: (11844, 57060)

Title: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট
সিন, ট্রায়াথলন, নদীতে, ইভেন্ট, দূষণ, স্থগিত, নদীর, দূষণমুক্ত, নদীকে, দূষণের

Title: এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!
এসিসির, মহসিন, নাকভি, সভাপতি, হচ্ছেন, ক্রিকেট, এশিয়ান, চেয়ারম্যানই, নাকভির, কাউন্সিলেরও

Title: ২০৩০ ও ২০৩৪ বিশ্বকাপের হোস্ট জানা যাবে দুইদিন পর!
হোস্ট, দুইদিন, হবার, বিশ্বকাপ, বিশ্বকাপের, বিড, ডেটলাইন, ফিফা, মরক্কো, আসর


In [43]:
# Build title-weighted predictions for the same 50 evaluation articles
eval_ids = set(manual_df["article_id"])

title_prediction_df = pd.DataFrame({
    "article_id": df_final.index,
    "title_weighted_keywords": [
        [word for word, score in article_keywords]
        for article_keywords in title_weighted_keywords
    ]
})

title_eval = manual_df.merge(
    title_prediction_df,
    on="article_id",
    how="left",
    validate="one_to_one"
)

title_eval["baseline_keywords"] = title_eval["article_id"].map(
    pd.Series(
        [ [word for word, score in article_keywords]
          for article_keywords in keywords ],
        index=df_final.index
    )
)

baseline_scores = title_eval.apply(
    lambda row: calculate_metrics(
        row["reference_keywords"],
        row["baseline_keywords"]
    ),
    axis=1
)

title_scores = title_eval.apply(
    lambda row: calculate_metrics(
        row["reference_keywords"],
        row["title_weighted_keywords"]
    ),
    axis=1
)

print("Baseline mean matched keywords:",
      baseline_scores["matched_keywords"].mean())

print("Title-weighted mean matched keywords:",
      title_scores["matched_keywords"].mean())

Baseline mean matched keywords: 3.26
Title-weighted mean matched keywords: 4.22


In [44]:
def summarize_scores(scores):
    return (
        scores[
            ["precision_at_10", "recall_at_10", "f1_at_10"]
        ].mean() * 100
    )

comparison = pd.DataFrame({
    "Baseline TF-IDF": summarize_scores(baseline_scores),
    "Title-weighted TF-IDF": summarize_scores(title_scores),
}).T.round(2)

display(comparison)

,precision_at_10,recall_at_10,f1_at_10
Baseline TF-IDF,32.6,32.6,32.6
Title-weighted TF-IDF,42.2,42.2,42.2


In [45]:
title_eval = title_eval.copy()
title_eval["baseline_matches"] = baseline_scores["matched_keywords"].values
title_eval["title_weighted_matches"] = title_scores["matched_keywords"].values

category_comparison = (
    title_eval.groupby("category")[
        ["baseline_matches", "title_weighted_matches"]
    ].mean().round(2)
)

display(category_comparison)

,baseline_matches,title_weighted_matches
category,,
entertainment,2.92,4.08
international,3.23,4.62
national,3.42,4.75
sports,3.46,3.46


In [46]:
# Reuse the title-weighted documents and remove BNLP stopwords.
from bnlp import BengaliCorpus

stopword_set = set(BengaliCorpus.stopwords)

weighted_documents_no_stopwords = []

for document in weighted_documents:
    tokens = document.split()
    filtered_tokens = [
        token for token in tokens
        if token not in stopword_set
    ]
    weighted_documents_no_stopwords.append(
        " ".join(filtered_tokens)
    )

stopword_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=100000,
)

stopword_matrix = stopword_vectorizer.fit_transform(
    weighted_documents_no_stopwords
)

stopword_keywords = extract_top_keywords(
    stopword_matrix,
    stopword_vectorizer.get_feature_names_out(),
    top_k=10,
)

print("Articles:", len(stopword_keywords))

for i in range(3):
    print("\nTitle:", df_final.iloc[i]["title"])
    print("Title-weighted:",
          ", ".join(w for w, s in title_weighted_keywords[i]))
    print("With stopword removal:",
          ", ".join(w for w, s in stopword_keywords[i]))

Articles: 11844

Title: সিন নদীতে ব্যাপক দূষণ, স্থগিত করা হলো ট্রায়াথলন ইভেন্ট
Title-weighted: সিন, ট্রায়াথলন, নদীতে, ইভেন্ট, দূষণ, স্থগিত, নদীর, দূষণমুক্ত, নদীকে, দূষণের
With stopword removal: সিন, ট্রায়াথলন, নদীতে, ইভেন্ট, দূষণ, স্থগিত, নদীর, দূষণমুক্ত, নদীকে, দূষণের

Title: এসিসির নতুন সভাপতি হচ্ছেন মহসিন নাকভি!
Title-weighted: এসিসির, মহসিন, নাকভি, সভাপতি, হচ্ছেন, ক্রিকেট, এশিয়ান, চেয়ারম্যানই, নাকভির, কাউন্সিলেরও
With stopword removal: এসিসির, মহসিন, নাকভি, সভাপতি, হচ্ছেন, ক্রিকেট, এশিয়ান, চেয়ারম্যানই, নাকভির, কাউন্সিলেরও

Title: ২০৩০ ও ২০৩৪ বিশ্বকাপের হোস্ট জানা যাবে দুইদিন পর!
Title-weighted: হোস্ট, দুইদিন, হবার, বিশ্বকাপ, বিশ্বকাপের, বিড, ডেটলাইন, ফিফা, মরক্কো, আসর
With stopword removal: হোস্ট, দুইদিন, হবার, বিশ্বকাপ, বিশ্বকাপের, বিড, ডেটলাইন, ফিফা, মরক্কো, আসর


In [47]:
# Prepare stopword-removal predictions for the same 50 articles
stopword_prediction_df = pd.DataFrame({
    "article_id": df_final.index,
    "stopword_keywords": [
        [word for word, score in article_keywords]
        for article_keywords in stopword_keywords
    ]
})

stopword_eval = manual_df.merge(
    stopword_prediction_df,
    on="article_id",
    how="left",
    validate="one_to_one"
)

stopword_scores = stopword_eval.apply(
    lambda row: calculate_metrics(
        row["reference_keywords"],
        row["stopword_keywords"]
    ),
    axis=1
)

stopword_summary = (
    stopword_scores[
        ["precision_at_10", "recall_at_10", "f1_at_10"]
    ].mean() * 100
)

print("Title-weighted + stopword removal")
print(f"Precision@10: {stopword_summary['precision_at_10']:.2f}%")
print(f"Recall@10:    {stopword_summary['recall_at_10']:.2f}%")
print(f"F1@10:        {stopword_summary['f1_at_10']:.2f}%")
print(
    "Average matched keywords:",
    f"{stopword_scores['matched_keywords'].mean():.2f} / 10"
)

Title-weighted + stopword removal
Precision@10: 42.20%
Recall@10:    42.20%
F1@10:        42.20%
Average matched keywords: 4.22 / 10


In [48]:
comparison = pd.DataFrame({
    "Baseline": summarize_scores(baseline_scores),
    "Title-weighted": summarize_scores(title_scores),
    "Title + stopwords": summarize_scores(stopword_scores),
}).T.round(2)

display(comparison)

,precision_at_10,recall_at_10,f1_at_10
Baseline,32.6,32.6,32.6
Title-weighted,42.2,42.2,42.2
Title + stopwords,42.2,42.2,42.2
